In [ ]:
import os, sys
if "google.colab" in sys.modules and not os.path.isdir("../data/raw"):
    if not os.path.isdir("/content/Busqueda-Empleo"):
        !git clone https://github.com/samuelvelez1003/Busqueda-Empleo.git /content/Busqueda-Empleo
    %cd /content/Busqueda-Empleo/notebooks
print("Carpeta de trabajo:", os.getcwd())

## Librerías y rutas

In [2]:
import os
import pandas as pd
import numpy as np

# Mostrar tablas completas (Colab y Jupyter recortan filas, columnas y textos largos)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

# Carpeta de datos del repositorio (el notebook se ejecuta desde /notebooks)
for d in ["../data", "data", "../Data", "Data"]:
    if os.path.isdir(d):
        DATA_DIR = d
        break
else:
    raise FileNotFoundError("No se encontró la carpeta data. Ejecute primero la celda de arriba (Colab) o abra el notebook desde la carpeta notebooks/")

RUTA_GEIH = os.path.join(DATA_DIR, "processed", "geih2025_jovenes_18_28.csv")
RUTA_SPE  = os.path.join(DATA_DIR, "raw", "spe", "11-Anexo-Demanda-Laboral-Ano-2023.xlsx")
SALIDA    = os.path.join(DATA_DIR, "processed")

print("GEIH:", RUTA_GEIH, "→ existe:", os.path.exists(RUTA_GEIH))
print("SPE :", RUTA_SPE,  "→ existe:", os.path.exists(RUTA_SPE))

GEIH: ../data/processed/geih2025_jovenes_18_28.csv → existe: True
SPE : ../data/raw/spe/11-Anexo-Demanda-Laboral-Ano-2023.xlsx → existe: True


## 1. Fuente 1 — GEIH 2025 (DANE)
**Qué es:** Gran Encuesta Integrada de Hogares, la encuesta oficial de mercado laboral de Colombia.  
**De dónde sale:** microdatos del DANE (https://microdatos.dane.gov.co/index.php/catalog/853), enero a diciembre de 2025.  
**Por qué la elegimos:** es la única fuente pública que dice, persona por persona, si un joven está ocupado o desocupado y **si ha trabajado antes**, que es justo nuestro problema.

Los 12 meses se unieron y limpiaron en `02_limpieza.ipynb` (4 módulos por mes → una fila por joven). Aquí leemos ese resultado.

In [3]:
geih = pd.read_csv(RUTA_GEIH)

print("Forma (filas, columnas):", geih.shape)
geih.head()

Forma (filas, columnas): (85422, 26)


,mes,id_hogar,orden,factor_expansion,codigo_departamento,departamento,ciudad,zona,edad,sexo,nivel_educativo,nivel_educativo_grupo,estudia_actualmente,condicion,ha_trabajado_antes,semanas_buscando,meses_en_empleo_actual,posicion_ocupacional,tiene_contrato,tipo_contrato,cotiza_pension,ingreso_laboral_mensual,sector,empleo_formal,sin_experiencia,consiguio_empleo
0,1,8086887-1,1,26.93,13,Bolívar,Resto del país,Rural y centros poblados,26,Mujer,Básica secundaria,Básica o menos,No,Ocupado,NaN,NaN,60.0,Cuenta propia,No,NaN,No,600000.0,Servicios administrativos y de apoyo,No,0,NaN
1,1,8086893-1,5,260.42,13,Bolívar,Resto del país,Urbana (cabecera),28,Hombre,Universitaria,Universitario,No,Ocupado,NaN,NaN,11.0,Cuenta propia,Sí,Término fijo,Sí,1123000.0,Educación,Sí,1,1.0
2,1,8086897-1,1,232.31,13,Bolívar,Resto del país,Urbana (cabecera),20,Mujer,Media académica,Bachiller,No,Desocupado,Sí,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,1,8086923-1,3,14.44,18,Caquetá,Resto del país,Rural y centros poblados,25,Mujer,Universitaria,Universitario,Sí,Ocupado,NaN,NaN,48.0,Trabajador familiar sin remuneración,No,NaN,No,NaN,Arte y entretenimiento,No,0,NaN
4,1,8086930-1,1,54.25,18,Caquetá,Resto del país,Rural y centros poblados,22,Hombre,Básica primaria,Básica o menos,No,Ocupado,NaN,NaN,1.0,Empleado empresa particular,Sí,NaN,No,1400000.0,Agropecuario,No,1,1.0


In [4]:
# Verificaciones rápidas
print("Rango de edad:", geih["edad"].min(), "-", geih["edad"].max())
print("Meses:", sorted(geih["mes"].unique()))
print("Departamentos distintos:", geih["codigo_departamento"].nunique())
print()
print(geih["condicion"].value_counts())
print()
print("Jóvenes sin experiencia (aspirantes + recién insertados):", geih["sin_experiencia"].sum())

Rango de edad: 18 - 28
Meses: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Departamentos distintos: 33

condicion
Ocupado       70168
Desocupado    15254
Name: count, dtype: int64

Jóvenes sin experiencia (aspirantes + recién insertados): 41785


## 2. Fuente 2 — Vacantes del Servicio Público de Empleo (SPE)
**Qué es:** *Anexo Estadístico de Demanda Laboral*: ofertas de empleo (vacantes) registradas en el SPE por mes, departamento y **experiencia requerida**, y por **nivel educativo requerido**.  
**De dónde sale:** https://www.serviciodeempleo.gov.co/dataempleo-spe/demanda-laboral/anexo-estadistico-de-demanda-laboral-vacantes/  
**Por qué la elegimos:** la GEIH describe a los jóvenes (la oferta de trabajo); el SPE describe **qué piden las empresas** (la demanda). Nos dice cuántas vacantes **no exigen experiencia** en cada departamento.

**Año usado:** 2023 (enero–noviembre), el más reciente que publica el SPE. Es una diferencia de año con la GEIH que documentamos como riesgo.

El archivo Excel tiene varias hojas; usamos **Experiencia** y **Educación** para los indicadores por departamento, **Sectores** para comparar sectores (sección 3.5) y **Municipios** para comparar ciudades (sección 3.6).

In [5]:
hojas = pd.ExcelFile(RUTA_SPE).sheet_names
print("Hojas del archivo:", hojas)

# Vista cruda de la hoja 'Experiencia' (tal como viene del SPE)
spe_crudo = pd.read_excel(RUTA_SPE, sheet_name="Experiencia", header=None)
print("Forma cruda:", spe_crudo.shape)
spe_crudo.iloc[4:12, :8]

Hojas del archivo: ['Índice', 'Departamentos', 'Municipios', 'Ocupaciones', 'Sectores', 'Cine', 'Educación', 'Prestador', 'Experiencia', 'Salarios', 'Notas']


Forma cruda: (322, 119)


,0,1,2,3,4,5,6,7
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Desagregación,NaN,NaN,2015,NaN,NaN,NaN,NaN
6,Código DIVIPOLA,Departamento,Experiencia laboral,Ene,Feb,Mar,Abr,May
7,0,Total nacional,Sin Experiencia,12276,11146,10573,11032,10626
8,0,Total nacional,1 a 6 Meses,1,0,0,2217,2638
9,0,Total nacional,7 a 12 Meses,22906,20984,22858,24592,22959
10,0,Total nacional,13 a 24 Meses,7445,7177,7372,8151,7873
11,0,Total nacional,25 a 36 Meses,3458,3310,3526,3458,3288


El Excel tiene formato de reporte: una fila con los **años**, otra con los **meses** y columnas para 2015 a 2023. La función `leer_hoja_spe` busca la fila de encabezados, se queda con la columna **Total 2023** y devuelve una tabla ordenada.

In [6]:
def leer_hoja_spe(hoja, anio=2023):
    """Lee una hoja del anexo del SPE y devuelve: código, departamento, categoría y total de vacantes del año."""
    crudo = pd.read_excel(RUTA_SPE, sheet_name=hoja, header=None)

    # 1) Fila donde empiezan los encabezados ("Código DIVIPOLA")
    fila_enc = crudo.index[crudo[0].astype(str).str.strip() == "Código DIVIPOLA"][0]

    # 2) Fila de años (justo arriba), rellenada hacia la derecha porque las celdas están combinadas
    anios = pd.to_numeric(crudo.loc[fila_enc - 1], errors="coerce").ffill()
    meses = crudo.loc[fila_enc].astype(str).str.strip()

    # 3) Columna 'Total' del año pedido
    col_total = [c for c in crudo.columns if anios[c] == anio and meses[c] == "Total"][0]

    datos = crudo.loc[fila_enc + 1:, [0, 1, 2, col_total]]
    datos.columns = ["codigo", "departamento_spe", "categoria", "vacantes"]
    datos = datos.dropna(subset=["departamento_spe", "categoria"])
    datos["categoria"] = datos["categoria"].astype(str).str.strip()
    datos["vacantes"] = pd.to_numeric(datos["vacantes"], errors="coerce").fillna(0).astype(int)

    # 4) Código DIVIPOLA numérico; 'Total nacional' y 'No especifica' no son departamentos
    datos["codigo_departamento"] = pd.to_numeric(datos["codigo"], errors="coerce")
    datos = datos[~datos["departamento_spe"].str.contains("Total nacional|No especifica", case=False)]
    datos = datos.dropna(subset=["codigo_departamento"])
    datos["codigo_departamento"] = datos["codigo_departamento"].astype(int)
    return datos.drop(columns="codigo")

spe_exp = leer_hoja_spe("Experiencia")
spe_edu = leer_hoja_spe("Educación")

print("Experiencia:", spe_exp.shape, "| Educación:", spe_edu.shape)
spe_exp.head(10)

Experiencia: (297, 4) | Educación: (363, 4)


,departamento_spe,categoria,vacantes,codigo_departamento
16,Amazonas,Sin Experiencia,189,91
17,Amazonas,1 a 6 Meses,274,91
18,Amazonas,7 a 12 Meses,288,91
19,Amazonas,13 a 24 Meses,685,91
20,Amazonas,25 a 36 Meses,69,91
21,Amazonas,37 a 60 Meses,127,91
22,Amazonas,Mayor a 60 meses,215,91
23,Amazonas,No especifica experiencia,38,91
24,Amazonas,Total Ofertas de empleo,1885,91
25,Antioquia,Sin Experiencia,52089,5


In [7]:
print("Categorías de experiencia:", spe_exp["categoria"].unique())
print("Categorías de educación:", spe_edu["categoria"].unique())
print("Departamentos en el SPE:", spe_exp["codigo_departamento"].nunique())

Categorías de experiencia: <StringArray>
[          'Sin Experiencia',               '1 a 6 Meses',
              '7 a 12 Meses',             '13 a 24 Meses',
             '25 a 36 Meses',             '37 a 60 Meses',
          'Mayor a 60 meses', 'No especifica experiencia',
   'Total Ofertas de empleo']
Length: 9, dtype: str
Categorías de educación: <StringArray>
[               'Primaria',              'Secundaria',
            'Bachillerato',                 'Técnica',
             'Tecnológica',           'Universitario',
         'Especialización',                'Maestría',
               'Doctorado',           'No especifica',
 'Total Ofertas de empleo']
Length: 11, dtype: str
Departamentos en el SPE: 33


### 2.1 Una fila por departamento
Pasamos de formato largo (departamento × categoría) a formato ancho y calculamos indicadores de la demanda laboral para jóvenes sin experiencia.

In [8]:
exp = spe_exp.pivot_table(index=["codigo_departamento", "departamento_spe"], columns="categoria",
                          values="vacantes", aggfunc="sum").reset_index()
edu = spe_edu.pivot_table(index="codigo_departamento", columns="categoria",
                          values="vacantes", aggfunc="sum").reset_index()
edu.columns = [str(c).strip() for c in edu.columns]

spe_depto = pd.DataFrame({
    "codigo_departamento": exp["codigo_departamento"],
    "departamento_spe": exp["departamento_spe"],
    "vacantes_total_2023": exp["Total Ofertas de empleo"],
    "vacantes_sin_experiencia_2023": exp["Sin Experiencia"],
    "vacantes_1_6_meses_2023": exp["1 a 6 Meses"],
})
spe_depto["pct_vacantes_sin_experiencia"] = (spe_depto["vacantes_sin_experiencia_2023"] / spe_depto["vacantes_total_2023"] * 100).round(1)
spe_depto["pct_vacantes_hasta_6_meses"] = ((spe_depto["vacantes_sin_experiencia_2023"] + spe_depto["vacantes_1_6_meses_2023"])
                                           / spe_depto["vacantes_total_2023"] * 100).round(1)

spe_depto = spe_depto.merge(edu[["codigo_departamento", "Bachillerato", "Técnica", "Tecnológica", "Universitario", "Total Ofertas de empleo"]],
                            on="codigo_departamento", how="left", validate="1:1")
tot_edu = spe_depto.pop("Total Ofertas de empleo")
spe_depto["pct_vacantes_bachillerato"] = (spe_depto.pop("Bachillerato") / tot_edu * 100).round(1)
spe_depto["pct_vacantes_tecnico_tecnologo"] = ((spe_depto.pop("Técnica") + spe_depto.pop("Tecnológica")) / tot_edu * 100).round(1)
spe_depto["pct_vacantes_universitario"] = (spe_depto.pop("Universitario") / tot_edu * 100).round(1)

print("Forma:", spe_depto.shape)
spe_depto.sort_values("vacantes_total_2023", ascending=False)

Forma: (33, 10)


,codigo_departamento,departamento_spe,vacantes_total_2023,vacantes_sin_experiencia_2023,vacantes_1_6_meses_2023,pct_vacantes_sin_experiencia,pct_vacantes_hasta_6_meses,pct_vacantes_bachillerato,pct_vacantes_tecnico_tecnologo,pct_vacantes_universitario
2,11,Bogotá D.C.,904252,126347,20227,14.0,16.2,32.1,28.7,22.8
0,5,Antioquia,373628,52089,13977,13.9,17.7,31.1,29.8,19.7
23,76,Valle del Cauca,175361,22153,6390,12.6,16.3,29.7,33.2,19.3
10,25,Cundinamarca,151193,17305,6017,11.4,15.4,35.5,26.4,14.4
20,68,Santander,99732,15028,3128,15.1,18.2,27.5,27.8,17.8
1,8,Atlántico,73897,9037,1513,12.2,14.3,21.8,31.5,31.2
15,50,Meta,61139,14704,2259,24.1,27.7,17.9,22.6,16.3
19,66,Risaralda,54832,9942,2191,18.1,22.1,39.7,26.7,12.8
3,13,Bolívar,38216,3458,1056,9.0,11.8,17.9,32.3,32.5
22,73,Tolima,29786,3567,2381,12.0,20.0,23.1,27.0,20.9


## 3. Cruce de las fuentes
**Columna en común:** `codigo_departamento` (código DIVIPOLA de 2 dígitos del DANE). Está en las dos fuentes porque ambas usan la codificación oficial de departamentos.

**Tipo de cruce:** `left` de GEIH (muchas personas) con SPE (una fila por departamento) → relación **muchos a uno** (`validate="m:1"`). Cada joven recibe los indicadores de demanda laboral de su departamento.

### 3.1 Verificación antes del cruce

In [9]:
filas_geih_antes = len(geih)
filas_spe = len(spe_depto)

print(f"Filas GEIH antes del cruce: {filas_geih_antes:,}")
print(f"Filas SPE (departamentos): {filas_spe}")
print("¿Código único en el SPE?:", spe_depto["codigo_departamento"].is_unique)

cod_geih = set(geih["codigo_departamento"].unique())
cod_spe = set(spe_depto["codigo_departamento"].unique())
print("Departamentos en GEIH:", len(cod_geih), "| en SPE:", len(cod_spe))
print("En GEIH pero no en SPE:", sorted(cod_geih - cod_spe))
print("En SPE pero no en GEIH:", sorted(cod_spe - cod_geih))

Filas GEIH antes del cruce: 85,422
Filas SPE (departamentos): 33
¿Código único en el SPE?: True
Departamentos en GEIH: 33 | en SPE: 33
En GEIH pero no en SPE: []
En SPE pero no en GEIH: []


### 3.2 El cruce (`pd.merge`)

In [10]:
cruzado = pd.merge(
    geih,
    spe_depto,
    on="codigo_departamento",
    how="left",
    validate="m:1",       # falla si el SPE tuviera departamentos repetidos
    indicator=True,       # agrega la columna _merge para verificar
)

print(f"Filas antes: {filas_geih_antes:,} | Filas después: {len(cruzado):,}")
print(cruzado["_merge"].value_counts())

Filas antes: 85,422 | Filas después: 85,422
_merge
both          85422
left_only         0
right_only        0
Name: count, dtype: int64


### 3.3 Verificación después del cruce

In [11]:
assert len(cruzado) == filas_geih_antes, "El cruce cambió el número de filas"
sin_pareja = (cruzado["_merge"] != "both").sum()
print("Filas de la GEIH sin pareja en el SPE:", sin_pareja)

# Los nombres de departamento de ambas fuentes deben coincidir
revision = cruzado[["codigo_departamento", "departamento", "departamento_spe"]].drop_duplicates().sort_values("codigo_departamento")
revision

Filas de la GEIH sin pareja en el SPE: 0


,codigo_departamento,departamento,departamento_spe
49,5,Antioquia,Antioquia
219,8,Atlántico,Atlántico
324,11,Bogotá D.C.,Bogotá D.C.
0,13,Bolívar,Bolívar
534,15,Boyacá,Boyacá
613,17,Caldas,Caldas
3,18,Caquetá,Caquetá
773,19,Cauca,Cauca
838,20,Cesar,Cesar
892,23,Córdoba,Córdoba


In [12]:
cruzado = cruzado.drop(columns=["_merge", "departamento_spe"])
print("Forma final del dataset cruzado:", cruzado.shape)
cruzado.head()

Forma final del dataset cruzado: (85422, 34)


,mes,id_hogar,orden,factor_expansion,codigo_departamento,departamento,ciudad,zona,edad,sexo,nivel_educativo,nivel_educativo_grupo,estudia_actualmente,condicion,ha_trabajado_antes,semanas_buscando,meses_en_empleo_actual,posicion_ocupacional,tiene_contrato,tipo_contrato,cotiza_pension,ingreso_laboral_mensual,sector,empleo_formal,sin_experiencia,consiguio_empleo,vacantes_total_2023,vacantes_sin_experiencia_2023,vacantes_1_6_meses_2023,pct_vacantes_sin_experiencia,pct_vacantes_hasta_6_meses,pct_vacantes_bachillerato,pct_vacantes_tecnico_tecnologo,pct_vacantes_universitario
0,1,8086887-1,1,26.93,13,Bolívar,Resto del país,Rural y centros poblados,26,Mujer,Básica secundaria,Básica o menos,No,Ocupado,NaN,NaN,60.0,Cuenta propia,No,NaN,No,600000.0,Servicios administrativos y de apoyo,No,0,NaN,38216,3458,1056,9.0,11.8,17.9,32.3,32.5
1,1,8086893-1,5,260.42,13,Bolívar,Resto del país,Urbana (cabecera),28,Hombre,Universitaria,Universitario,No,Ocupado,NaN,NaN,11.0,Cuenta propia,Sí,Término fijo,Sí,1123000.0,Educación,Sí,1,1.0,38216,3458,1056,9.0,11.8,17.9,32.3,32.5
2,1,8086897-1,1,232.31,13,Bolívar,Resto del país,Urbana (cabecera),20,Mujer,Media académica,Bachiller,No,Desocupado,Sí,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,38216,3458,1056,9.0,11.8,17.9,32.3,32.5
3,1,8086923-1,3,14.44,18,Caquetá,Resto del país,Rural y centros poblados,25,Mujer,Universitaria,Universitario,Sí,Ocupado,NaN,NaN,48.0,Trabajador familiar sin remuneración,No,NaN,No,NaN,Arte y entretenimiento,No,0,NaN,4095,666,464,16.3,27.6,27.6,22.2,12.4
4,1,8086930-1,1,54.25,18,Caquetá,Resto del país,Rural y centros poblados,22,Hombre,Básica primaria,Básica o menos,No,Ocupado,NaN,NaN,1.0,Empleado empresa particular,Sí,NaN,No,1400000.0,Agropecuario,No,1,1.0,4095,666,464,16.3,27.6,27.6,22.2,12.4


### 3.4 Vista por departamento
Resumen que conecta las dos fuentes: tasa ponderada con la que los jóvenes sin experiencia consiguen empleo (GEIH) frente al porcentaje de vacantes que no exigen experiencia (SPE).

In [13]:
base = cruzado[cruzado["sin_experiencia"] == 1]

resumen = (base.groupby(["codigo_departamento", "departamento"])
               .apply(lambda g: pd.Series({
                   "jovenes_sin_exp_muestra": len(g),
                   "tasa_empleo_sin_exp_%": np.average(g["consiguio_empleo"], weights=g["factor_expansion"]) * 100,
                   "tasa_empleo_formal_%": np.average((g["empleo_formal"] == "Sí"), weights=g["factor_expansion"]) * 100,
               }), include_groups=False)
               .reset_index()
               .merge(spe_depto.drop(columns="departamento_spe"), on="codigo_departamento", how="left", validate="1:1")
               .round(1)
               .sort_values("pct_vacantes_sin_experiencia", ascending=False))

print("Departamentos en el resumen:", len(resumen))
resumen

Departamentos en el resumen: 33


,codigo_departamento,departamento,jovenes_sin_exp_muestra,tasa_empleo_sin_exp_%,tasa_empleo_formal_%,vacantes_total_2023,vacantes_sin_experiencia_2023,vacantes_1_6_meses_2023,pct_vacantes_sin_experiencia,pct_vacantes_hasta_6_meses,pct_vacantes_bachillerato,pct_vacantes_tecnico_tecnologo,pct_vacantes_universitario
26,86,Putumayo,224.0,87.1,39.3,6135,2717,269,44.3,48.7,11.0,18.8,10.3
27,88,San Andrés,371.0,89.0,79.1,3512,1133,207,32.3,38.2,21.3,21.2,5.5
25,85,Casanare,360.0,95.6,39.4,27636,8828,919,31.9,35.3,13.4,17.7,8.8
24,81,Arauca,237.0,80.8,32.6,6048,1878,365,31.1,37.1,9.6,16.3,12.6
15,50,Meta,2206.0,95.4,37.0,61139,14704,2259,24.1,27.7,17.9,22.6,16.3
8,20,Cesar,1187.0,91.0,20.6,15561,3036,999,19.5,25.9,19.9,24.4,17.4
9,23,Córdoba,1329.0,86.4,9.7,10969,2023,473,18.4,22.8,21.1,27.7,18.3
19,66,Risaralda,2105.0,94.3,48.9,54832,9942,2191,18.1,22.1,39.7,26.7,12.8
4,15,Boyacá,1505.0,92.3,34.3,28544,4951,2403,17.3,25.8,21.4,23.4,18.5
21,70,Sucre,1297.0,83.6,10.8,6084,1002,382,16.5,22.7,19.8,23.0,15.8


In [14]:
corr = resumen[["tasa_empleo_sin_exp_%", "tasa_empleo_formal_%", "pct_vacantes_sin_experiencia",
                "pct_vacantes_tecnico_tecnologo"]].corr().round(2)
corr

,tasa_empleo_sin_exp_%,tasa_empleo_formal_%,pct_vacantes_sin_experiencia,pct_vacantes_tecnico_tecnologo
tasa_empleo_sin_exp_%,1.00,0.37,0.07,0.21
tasa_empleo_formal_%,0.37,1.00,0.23,0.08
pct_vacantes_sin_experiencia,0.07,0.23,1.00,-0.63
pct_vacantes_tecnico_tecnologo,0.21,0.08,-0.63,1.00


### 3.5 Sectores: dónde se emplean los jóvenes y dónde están las vacantes
La hoja **Sectores** del SPE trae las vacantes por sección CIIU Rev. 4 (A, B, C…) y departamento. Cada sección se asigna al mismo grupo de sectores que usa `02_limpieza` para la GEIH.

- **GEIH:** en qué sectores trabajan los jóvenes **recién insertados** (`consiguio_empleo = 1`), ponderado con el factor de expansión.
- **SPE:** qué porcentaje de las vacantes de 2023 corresponde a cada sector.

**Ojo:** en la GEIH el sector solo existe para quienes ya tienen empleo, así que se usa de forma **descriptiva** y no como variable del modelo. Además, el SPE no cruza sector con experiencia requerida: la tabla muestra dónde hay más vacantes en general, no cuáles aceptan personas sin experiencia.

In [15]:
# Sección CIIU del SPE → sector de la GEIH (misma agrupación de 02_limpieza)
SECCION_A_SECTOR = {
    "A": "Agropecuario", "B": "Minería", "C": "Industria manufacturera", "D": "Servicios públicos",
    "E": "Servicios públicos", "F": "Construcción", "G": "Comercio", "H": "Transporte y logística",
    "I": "Alojamiento y comida", "J": "Información y comunicaciones (TIC)", "K": "Servicios financieros",
    "L": "Inmobiliarias", "M": "Actividades profesionales", "N": "Servicios administrativos y de apoyo",
    "O": "Administración pública", "P": "Educación", "Q": "Salud", "R": "Arte y entretenimiento",
    "S": "Otros servicios", "T": "Hogares como empleadores", "U": "Organismos extraterritoriales"}

spe_sec = leer_hoja_spe("Sectores")
spe_sec["sector"] = spe_sec["categoria"].str.replace("SECCIÓN", "").str.strip().map(SECCION_A_SECTOR)
# N.S. = no especifica sector; Total Ofertas de empleo = subtotal. Ninguna de las dos se usa
print("Categorías del SPE que no se usan:", list(spe_sec.loc[spe_sec["sector"].isna(), "categoria"].unique()))

insertados = cruzado[cruzado["consiguio_empleo"] == 1]

sectores = pd.DataFrame({
    "jovenes_insertados_estimados": insertados.groupby("sector")["factor_expansion"].sum(),
    "vacantes_spe_2023": spe_sec.groupby("sector")["vacantes"].sum(),
}).fillna(0)
sectores["pct_jovenes_insertados"] = sectores["jovenes_insertados_estimados"] / sectores["jovenes_insertados_estimados"].sum() * 100
sectores["pct_vacantes_spe"] = sectores["vacantes_spe_2023"] / sectores["vacantes_spe_2023"].sum() * 100
# > 0: el sector emplea más jóvenes de lo que pesa en las vacantes del SPE (empleo que no pasa por el SPE, informalidad…)
sectores["diferencia_pp"] = sectores["pct_jovenes_insertados"] - sectores["pct_vacantes_spe"]
sectores = (sectores.rename_axis("sector").reset_index()
                    .sort_values("pct_jovenes_insertados", ascending=False)
                    .round(1))

print(f"Vacantes del SPE 2023 con sector: {sectores['vacantes_spe_2023'].sum():,.0f}")
sectores

Categorías del SPE que no se usan: ['N.S.', 'Total Ofertas de empleo']
Vacantes del SPE 2023 con sector: 2,017,488


,sector,jovenes_insertados_estimados,vacantes_spe_2023,pct_jovenes_insertados,pct_vacantes_spe,diferencia_pp
5,Comercio,503394.8,323941,19.3,16.1,3.2
3,Alojamiento y comida,319255.7,66902,12.2,3.3,8.9
9,Industria manufacturera,296531.4,251518,11.3,12.5,-1.1
2,Agropecuario,265381.8,22913,10.1,1.1,9.0
19,Transporte y logística,184490.6,108440,7.1,5.4,1.7
6,Construcción,174636.9,113600,6.7,5.6,1.0
16,Servicios administrativos y de apoyo,136813.7,228120,5.2,11.3,-6.1
15,Salud,121353.5,158371,4.6,7.8,-3.2
14,Otros servicios,106322.6,113939,4.1,5.6,-1.6
7,Educación,80662.3,71595,3.1,3.5,-0.5


### 3.6 Ciudades: vacantes en la capital frente a la tasa de inserción
La hoja **Municipios** del SPE trae las vacantes por municipio (solo el total, sin experiencia requerida). Para cada una de las 32 ciudades de la GEIH se toma su municipio capital, cuyo código DIVIPOLA es `código del departamento × 1000 + 1` (por ejemplo, Medellín = 05001).

Para comparar ciudades de distinto tamaño se calculan las **vacantes registradas en 2023 por cada 1.000 jóvenes** de 18 a 28 años en la fuerza de trabajo (GEIH, con factor de expansión). En las áreas metropolitanas (Medellín A.M., Cali A.M., …) solo se cuentan las vacantes de la capital, así que su valor queda algo subestimado.

Sirve para revisar la **hipótesis secundaria**: las ciudades con más vacantes por joven tienen mayor tasa de inserción.

In [16]:
muni = pd.read_excel(RUTA_SPE, sheet_name="Municipios", header=None)

# En esta hoja los años están en la fila de "Código DIVIPOLA" y los meses en la fila siguiente
fila_anios = muni.index[muni[0].astype(str).str.contains("DIVIPOLA")][0]
anios_m = pd.to_numeric(muni.loc[fila_anios], errors="coerce").ffill()
meses_m = muni.loc[fila_anios + 1].astype(str).str.strip()
col_total_m = [c for c in muni.columns if anios_m[c] == 2023 and meses_m[c] == "Total"][0]

vac_muni = muni.loc[fila_anios + 2:, [0, 2, col_total_m]]
vac_muni.columns = ["codigo_municipio", "municipio_spe", "vacantes_capital_2023"]
vac_muni["codigo_municipio"] = pd.to_numeric(vac_muni["codigo_municipio"], errors="coerce")
vac_muni["vacantes_capital_2023"] = pd.to_numeric(vac_muni["vacantes_capital_2023"], errors="coerce").round().astype("Int64")
vac_muni = vac_muni.dropna(subset=["codigo_municipio"])
vac_muni = vac_muni[vac_muni["codigo_municipio"] > 0]   # 0 = Total nacional

# GEIH: jóvenes en la fuerza de trabajo y tasa de inserción de los sin experiencia, por ciudad
en_ciudad = cruzado[cruzado["ciudad"] != "Resto del país"]
assert (en_ciudad.groupby("ciudad")["codigo_departamento"].nunique() == 1).all()

ciudades = en_ciudad.groupby("ciudad").agg(
    codigo_departamento=("codigo_departamento", "first"),
    jovenes_fuerza_trabajo_estimados=("factor_expansion", "sum"))
ciudades["tasa_empleo_sin_exp_%"] = (en_ciudad[en_ciudad["sin_experiencia"] == 1]
                                     .groupby("ciudad")
                                     .apply(lambda g: np.average(g["consiguio_empleo"], weights=g["factor_expansion"]) * 100,
                                            include_groups=False))
ciudades["codigo_municipio"] = ciudades["codigo_departamento"] * 1000 + 1

ciudades = ciudades.reset_index().merge(vac_muni, on="codigo_municipio", how="left", validate="1:1")
ciudades["vacantes_por_1000_jovenes"] = ciudades["vacantes_capital_2023"] / ciudades["jovenes_fuerza_trabajo_estimados"] * 1000
ciudades = ciudades.round(1).sort_values("vacantes_por_1000_jovenes", ascending=False)

print("Ciudades:", len(ciudades), "| sin municipio en el SPE:", ciudades["vacantes_capital_2023"].isna().sum())
ciudades

Ciudades: 32 | sin municipio en el SPE: 0


,ciudad,codigo_departamento,jovenes_fuerza_trabajo_estimados,tasa_empleo_sin_exp_%,codigo_municipio,municipio_spe,vacantes_capital_2023,vacantes_por_1000_jovenes
3,Bogotá,11,1016301.0,93.8,11001,"BOGOTÁ, D.C.",904252,889.7
24,San Andrés,88,3853.0,89.0,88001,SAN ANDRÉS,3350,869.4
11,Leticia,91,2783.6,76.4,91001,LETICIA,1824,655.3
28,Tunja,15,17935.2,90.2,15001,TUNJA,11594,646.4
19,Pereira A.M.,66,77693.9,94.7,66001,PEREIRA,48440,623.5
31,Yopal,85,22378.5,95.6,85001,YOPAL,12176,544.1
17,Neiva,41,32637.5,87.0,41001,NEIVA,16886,517.4
13,Medellín A.M.,5,546211.3,94.9,5001,MEDELLÍN,268838,492.2
5,Cali A.M.,76,272600.7,92.1,76001,CALI,128850,472.7
30,Villavicencio,50,65064.9,96.1,50001,VILLAVICENCIO,28443,437.1


In [17]:
# Correlación de Spearman (por rangos): no asume relación lineal y es menos sensible a Bogotá, que es un valor extremo
ciudades[["tasa_empleo_sin_exp_%", "vacantes_por_1000_jovenes"]].corr(method="spearman").round(2)

,tasa_empleo_sin_exp_%,vacantes_por_1000_jovenes
tasa_empleo_sin_exp_%,1.00,0.25
vacantes_por_1000_jovenes,0.25,1.00


## 4. Diccionario de datos del dataset cruzado

In [18]:
DESCRIPCION = {
    "mes": ("Mes de la encuesta (1 = enero … 12 = diciembre de 2025)", "GEIH", "MES"),
    "id_hogar": ("Identificador anonimizado del hogar (DIRECTORIO-SECUENCIA_P)", "GEIH", "DIRECTORIO, SECUENCIA_P"),
    "orden": ("Número de la persona dentro del hogar", "GEIH", "ORDEN"),
    "factor_expansion": ("Peso de la persona para estimar la población (FEX_C18 / 12)", "GEIH", "FEX_C18"),
    "codigo_departamento": ("Código DIVIPOLA del departamento. LLAVE DEL CRUCE", "GEIH y SPE", "DPTO / Código DIVIPOLA"),
    "departamento": ("Nombre del departamento", "GEIH", "DPTO"),
    "ciudad": ("Ciudad o área metropolitana (Resto del país si no es una de las 32 principales)", "GEIH", "AREA"),
    "zona": ("Urbana (cabecera) o rural y centros poblados", "GEIH", "CLASE"),
    "edad": ("Edad en años cumplidos (18 a 28)", "GEIH", "P6040"),
    "sexo": ("Sexo al nacer", "GEIH", "P3271"),
    "nivel_educativo": ("Mayor nivel educativo alcanzado", "GEIH", "P3042"),
    "nivel_educativo_grupo": ("Nivel educativo agrupado (Básica o menos, Bachiller, Técnico, Tecnólogo, Universitario, Posgrado)", "GEIH", "P3042"),
    "estudia_actualmente": ("¿Asiste actualmente a una institución educativa?", "GEIH", "P6170"),
    "condicion": ("Ocupado o Desocupado", "GEIH", "Módulos Ocupados / No ocupados (DSI)"),
    "ha_trabajado_antes": ("Solo desocupados: ¿ha trabajado alguna vez al menos 2 semanas seguidas?", "GEIH", "P7430"),
    "semanas_buscando": ("Solo desocupados: semanas buscando trabajo", "GEIH", "P7250"),
    "meses_en_empleo_actual": ("Solo ocupados: meses trabajando de forma continua en el empleo actual", "GEIH", "P6426"),
    "posicion_ocupacional": ("Solo ocupados: empleado particular, cuenta propia, doméstico, etc.", "GEIH", "P6430"),
    "tiene_contrato": ("Solo ocupados: ¿tiene contrato (verbal o escrito)?", "GEIH", "P6440"),
    "tipo_contrato": ("Solo ocupados con contrato: término indefinido o fijo", "GEIH", "P6460"),
    "cotiza_pension": ("Solo ocupados: ¿cotiza a un fondo de pensiones?", "GEIH", "P6920"),
    "ingreso_laboral_mensual": ("Solo ocupados: ingreso laboral mensual en pesos", "GEIH", "INGLABO"),
    "sector": ("Solo ocupados: sector económico del empleo (CIIU Rev. 4 agrupado)", "GEIH", "RAMA2D_R4"),
    "empleo_formal": ("Solo ocupados: Sí si cotiza a pensión (aproximación de formalidad)", "GEIH (calculada)", "P6920"),
    "sin_experiencia": ("1 = aspirante (desocupado que nunca ha trabajado) o recién insertado (ocupado ≤ 12 meses en su empleo)", "GEIH (calculada)", "P7430, P6426"),
    "consiguio_empleo": ("VARIABLE OBJETIVO. 1 = recién insertado, 0 = aspirante; vacío si sin_experiencia = 0", "GEIH (calculada)", "P7430, P6426"),
    "vacantes_total_2023": ("Total de vacantes registradas en el SPE en el departamento (ene–nov 2023)", "SPE", "Hoja Experiencia: Total Ofertas de empleo"),
    "vacantes_sin_experiencia_2023": ("Vacantes que no exigen experiencia en el departamento (ene–nov 2023)", "SPE", "Hoja Experiencia: Sin Experiencia"),
    "vacantes_1_6_meses_2023": ("Vacantes que piden de 1 a 6 meses de experiencia (ene–nov 2023)", "SPE", "Hoja Experiencia: 1 a 6 Meses"),
    "pct_vacantes_sin_experiencia": ("% de las vacantes del departamento que no exigen experiencia", "SPE (calculada)", "Hoja Experiencia"),
    "pct_vacantes_hasta_6_meses": ("% de vacantes que piden 6 meses o menos de experiencia", "SPE (calculada)", "Hoja Experiencia"),
    "pct_vacantes_bachillerato": ("% de vacantes que piden bachillerato", "SPE (calculada)", "Hoja Educación"),
    "pct_vacantes_tecnico_tecnologo": ("% de vacantes que piden formación técnica o tecnológica", "SPE (calculada)", "Hoja Educación"),
    "pct_vacantes_universitario": ("% de vacantes que piden formación universitaria", "SPE (calculada)", "Hoja Educación"),
}

diccionario = pd.DataFrame([
    {"columna": col,
     "tipo_dato": str(cruzado[col].dtype),
     "significado": DESCRIPCION[col][0],
     "fuente": DESCRIPCION[col][1],
     "variable_original": DESCRIPCION[col][2],
     "% vacío": round(cruzado[col].isna().mean() * 100, 1),
     "ejemplo": cruzado[col].dropna().iloc[0] if cruzado[col].notna().any() else None}
    for col in cruzado.columns
])

faltan = [c for c in cruzado.columns if c not in DESCRIPCION]
print("Columnas sin descripción:", faltan)
diccionario

Columnas sin descripción: []


,columna,tipo_dato,significado,fuente,variable_original,% vacío,ejemplo
0,mes,int64,Mes de la encuesta (1 = enero … 12 = diciembre de 2025),GEIH,MES,0.0,1
1,id_hogar,str,Identificador anonimizado del hogar (DIRECTORIO-SECUENCIA_P),GEIH,"DIRECTORIO, SECUENCIA_P",0.0,8086887-1
2,orden,int64,Número de la persona dentro del hogar,GEIH,ORDEN,0.0,1
3,factor_expansion,float64,Peso de la persona para estimar la población (FEX_C18 / 12),GEIH,FEX_C18,0.0,26.93
4,codigo_departamento,int64,Código DIVIPOLA del departamento. LLAVE DEL CRUCE,GEIH y SPE,DPTO / Código DIVIPOLA,0.0,13
5,departamento,str,Nombre del departamento,GEIH,DPTO,0.0,Bolívar
6,ciudad,str,Ciudad o área metropolitana (Resto del país si no es una de las 32 principales),GEIH,AREA,0.0,Resto del país
7,zona,str,Urbana (cabecera) o rural y centros poblados,GEIH,CLASE,0.0,Rural y centros poblados
8,edad,int64,Edad en años cumplidos (18 a 28),GEIH,P6040,0.0,26
9,sexo,str,Sexo al nacer,GEIH,P3271,0.0,Mujer


## 5. Guardar resultados
- `dataset_cruzado_sin_experiencia.csv`: jóvenes **sin experiencia** (la población de la pregunta) con las variables de ambas fuentes. Es la base para el análisis y el modelo.
- `resumen_departamentos_geih_spe.csv`: una fila por departamento.
- `resumen_sectores_geih_spe.csv`: una fila por sector (sección 3.5).
- `resumen_ciudades_geih_spe.csv`: una fila por ciudad (sección 3.6).
- `data/diccionario/diccionario_dataset_cruzado.csv`: el diccionario de arriba.

In [19]:
final = cruzado[cruzado["sin_experiencia"] == 1]
final.to_csv(os.path.join(SALIDA, "dataset_cruzado_sin_experiencia.csv"), index=False, encoding="utf-8-sig")
resumen.to_csv(os.path.join(SALIDA, "resumen_departamentos_geih_spe.csv"), index=False, encoding="utf-8-sig")
sectores.to_csv(os.path.join(SALIDA, "resumen_sectores_geih_spe.csv"), index=False, encoding="utf-8-sig")
ciudades.to_csv(os.path.join(SALIDA, "resumen_ciudades_geih_spe.csv"), index=False, encoding="utf-8-sig")
diccionario.to_csv(os.path.join(DATA_DIR, "diccionario", "diccionario_dataset_cruzado.csv"), index=False, encoding="utf-8-sig")

print("Dataset cruzado (sin experiencia):", final.shape)
print("Resumen por departamento:", resumen.shape)
print("Resumen por sector:", sectores.shape)
print("Resumen por ciudad:", ciudades.shape)

Dataset cruzado (sin experiencia): (41785, 34)
Resumen por departamento: (33, 13)
Resumen por sector: (20, 6)
Resumen por ciudad: (32, 8)
